# Warning: Due to space constraints from running my tests I clear out the zip and unzipped folders. Ensure you have the zip stored in a separate location for additional operations.

# Note it was found that I could not easily dynamically pull the notebook name so if you update the notebook update the following code:

In [1]:
# Run the following once and once only when setting up jupyter notebooks for the 12-xx-xx series tests
# sudo apt install -y python3-dmidecode
# !pip3 install torch==2.8.0+cu124 --index-url https://download.pytorch.org/whl/cu124

In [2]:
nb_name = "MAT-12-02-level-2-notebook"

In [3]:
# Native
import concurrent.futures
from datetime import datetime
import hashlib
import os
from pathlib import Path
import shutil
import sys
import time
import zipfile

# Third Party
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from tqdm import tqdm

/home/flaniganp/miniconda3/envs/my-python-buddy-notebook/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Confirm you are using the GPU method
def print_gpu_info():
    print("GPU Characteristics Printout")

    # Is CUDA available?
    print("CUDA available:", torch.cuda.is_available())
    print("Device count:", torch.cuda.device_count())

    if torch.cuda.is_available():
        idx = torch.cuda.current_device()
        print("Current device index:", idx)
        print("Device name:", torch.cuda.get_device_name(idx))
        print("Total memory (GB):", round(torch.cuda.get_device_properties(idx).total_memory / 1e9, 2))
        print("Multiprocessors:", torch.cuda.get_device_properties(idx).multi_processor_count)
        print("Compute capability:", torch.cuda.get_device_properties(idx).major, ".", torch.cuda.get_device_properties(idx).minor)
        print("CUDA Runtime:", torch.version.cuda)

In [5]:
# Exercise GPU Printout
print_gpu_info()

GPU Characteristics Printout
CUDA available: True
Device count: 1
Current device index: 0
Device name: NVIDIA GeForce RTX 3090
Total memory (GB): 25.41
Multiprocessors: 82
Compute capability: 8 . 6
CUDA Runtime: 12.8


In [6]:
# Get project base directory (one level up from current working directory)
base_dir = Path.cwd().parent

base_application_dir = base_dir / "base_application"

# Convert to absolute string path
base_application_dir = str(base_application_dir.resolve())

# Add to Python path if not already present
if base_application_dir not in sys.path:
    sys.path.append(base_application_dir)

print("base_application added to PATH:")
print(base_application_dir)

base_application added to PATH:
/home/flaniganp/Documents/my-python-buddy/base_application


In [7]:
from views_chat_utilities import SYSTEM_PROMPTS

In [8]:
# Generates a cleaned text response using a Hugging Face model pipeline (non-llama specific).
def get_cleaned_code_response_merged_model(llm_model, tokenizer, user_question):
    # Normalize system prompt
    system_prompt = SYSTEM_PROMPTS["code_writer"]

    # Build conversation structure
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_question},
    ]

    # Attempt to build a structured prompt (chat template if available)
    try:
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    except Exception:
        bos = tokenizer.bos_token or ""
        prompt = f"{bos}[INST] <<SYS>>\n{system_prompt}\n<</SYS>>\n\n{user_question} [/INST]"

    # Configure model for inference
    llm_model.eval()
    try:
        # Some models may not support gradient checkpointing disable
        llm_model.gradient_checkpointing_disable()  # noqa: B110
    except AttributeError:
        # Attribute may not exist on all model types and is safe to ignore
        pass
    llm_model.config.use_cache = True

    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # Generation settings
    max_new_tokens = 300
    generation_kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.05,
        no_repeat_ngram_size=6,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        return_full_text=False,
    )

    # Create pipeline and generate
    text_pipe = pipeline(
        task="text-generation",
        model=llm_model,
        tokenizer=tokenizer,
        device_map="auto",
    )

    result = text_pipe(prompt, **generation_kwargs)

    # Extract generated text and token counts
    generated_text = result[0]["generated_text"].strip()
    prompt_tokens = len(tokenizer.encode(prompt))
    max_tokens_used = max_new_tokens

    # Postprocess text (placeholder for cleaning or formatting)
    cleaned_text = generated_text.strip()

    return cleaned_text, prompt_tokens, max_tokens_used

In [9]:
# Compute SHA256 of a file
def sha256sum(file_path, block_size=65536):
    sha = hashlib.sha256()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(block_size), b""):
            sha.update(chunk)
    return sha.hexdigest()

In [10]:
# Define paths
zip_path = os.path.join(base_dir, "models", "llama-2-7b-143k-codeAlpaca-2025-10-30_1326.zip")
hash_path = os.path.join(base_dir, "models", "llama-2-merged-7b-143k-codeAlpaca-2025-10-30_1326-hash.txt")
extract_dir = os.path.join(base_dir, "models", "llama-2-merged-7b-143k-codeAlpaca-2025-10-30_1143")

In [11]:
if os.path.exists(extract_dir):
    try:
        shutil.rmtree(extract_dir)
        print(f"Successfully deleted directory: {extract_dir}")
    except Exception as e:
        print(f"Error deleting {extract_dir}: {e}")
else:
    print(f"Directory not found: {extract_dir}.")

Directory not found: /home/flaniganp/Documents/my-python-buddy/models/llama-2-merged-7b-143k-codeAlpaca-2025-10-30_1143.


In [12]:
# Read expected hash
with open(hash_path, "r") as f:
    expected_hash = f.read().strip()

# Compute actual hash
actual_hash = sha256sum(zip_path)

# Compare
if actual_hash == expected_hash:
    print(f"Hash verified: {actual_hash}")
else:
    raise Exception((f"Hash mismatch!\nExpected: {expected_hash}\nFound: {actual_hash}"))

Hash verified: a7313874122da306d722fa8d77854a93cba398576f1a9b240830fbb6723e180b


In [13]:
if not os.path.exists(extract_dir):
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        members = zf.infolist()

        # Define a worker that extracts one file at a time
        def extract_member(member):
            zf.extract(member, extract_dir)

        # Use ThreadPoolExecutor (I/O bound) instead of ProcessPoolExecutor
        # because ZipFile objects aren’t pickle-safe
        with concurrent.futures.ThreadPoolExecutor() as executor:
            list(
                tqdm(
                    executor.map(extract_member, members),
                    total=len(members),
                    desc="Extracting",
                )
            )
    print(f"Extracted to: {extract_dir}")
else:
    print(f"Directory already exists: {extract_dir}")

Extracting: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 12/12 [00:19<00:00,  1.66s/it]

Extracted to: /home/flaniganp/Documents/my-python-buddy/models/llama-2-merged-7b-143k-codeAlpaca-2025-10-30_1143


In [14]:
# Remove archive (comment out if you want to keep it)
if os.path.exists(zip_path):
    try:
        os.remove(zip_path)
        print(f"Successfully deleted archive: {zip_path}")
    except Exception as e:
        print(f"Error deleting file {zip_path}: {e}")
else:
    print(f"File not found: {zip_path}")

Successfully deleted archive: /home/flaniganp/Documents/my-python-buddy/models/llama-2-7b-143k-codeAlpaca-2025-10-30_1326.zip


In [15]:
# Load model and tokenizer
print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(extract_dir, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(extract_dir)

# Confirm load
print(f"Model and tokenizer loaded from {extract_dir}")

/home/flaniganp/miniconda3/envs/my-python-buddy-notebook/lib/python3.10/site-packages/accelerate/utils/modeling.py:1566: UserWarning: Current model requires 256 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(


Loading model...


Loading checkpoint shards: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:01<00:00,  1.55it/s]

Model and tokenizer loaded from /home/flaniganp/Documents/my-python-buddy/models/llama-2-merged-7b-143k-codeAlpaca-2025-10-30_1143


In [16]:
questions = [
    "Code: How do I filter even numbers from a list using a list comprehension?",
    "Code: How do I find the common elements between two lists in Python?",
    "Code: How do I remove all vowels from a string in Python?",
    "Code: How do I count how many times each word appears in a string?",
    "Code: How do I check if a string is a palindrome, ignoring case and spaces?",
    "Code: How do I get the current date and time in Python?",
    "Code: How do I find the factorial of a number using recursion in Python?",
    "Code: How do I merge two dictionaries into one in Python?",
    "Code: How do I find all unique words in a sentence, ignoring case and punctuation?",
    "Code: How do I find the most frequent element in a list without using the collections module?",
    "Code: How do I determine if two strings are anagrams of each other in Python?",
]

In [17]:
# Setup output directory and file
base_dir = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
results_dir = base_dir / "test_results"
results_dir.mkdir(exist_ok=True)

timestamp = datetime.now().strftime("%Y_%m_%d_%H_%M_%S")
output_file = results_dir / f"{nb_name}_{timestamp}.txt"

execution_times = []  # store all durations

# Begin loop over questions
with open(output_file, "w", encoding="utf-8") as f:
    for i, user_question in enumerate(questions, start=1):
        print(f"\nQuestion {i}: {user_question}")
        f.write(f"\nQuestion {i}: {user_question}\n")

        # Measure time
        start_time = time.time()
        cleaned_response, prompt_tokens, max_new_tokens = get_cleaned_code_response_merged_model(model, tokenizer, user_question)
        end_time = time.time()
        duration = end_time - start_time
        execution_times.append(duration)

        # Display and record results
        print(f"Cleaned Response:\n{cleaned_response}")
        print("-" * 88)

        f.write(f"Cleaned Response:\n{cleaned_response}\n")
        f.write(f"Prompt Tokens: {prompt_tokens}\n")
        f.write(f"Max New Tokens: {max_new_tokens}\n")
        f.write(f"Time Taken: {duration:.2f} seconds\n")
        f.write("Is result correct? If not, one sentence as to why:\n\n")
        f.write("-" * 88 + "\n")

    # Compute and record summary stats
    if execution_times:
        low_time = min(execution_times)
        high_time = max(execution_times)
        avg_time = sum(execution_times) / len(execution_times)

        summary = (
            f"\nExecution Time Summary:\n"
            f"Lowest Time:  {low_time:.2f} seconds\n"
            f"Highest Time: {high_time:.2f} seconds\n"
            f"Average Time: {avg_time:.2f} seconds\n"
        )

        print(summary)
        f.write(summary)

print(f"\nAll results recorded to: {output_file}")

Device set to use cpu
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Question 1: Code: How do I filter even numbers from a list using a list comprehension?


Device set to use cpu


Cleaned Response:
```python
# Define a list of numbers
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# Filter out even numbers using list comprehension
filtered_numbers = [num for num in numbers if num % 2!= 0]

print(filtered_numbers)
```
----------------------------------------------------------------------------------------

Question 2: Code: How do I find the common elements between two lists in Python?


Device set to use cpu


...

Cleaned Response:
You can merge two dictionaries into a single dictionary in Python using the `update()` method. Here's an example:

```python
# Define two dictionaries
dict1 = {'a': 1, 'b': 2, 'c': 3}
dict2 = {'d': 4, 'e': 5, 'f': 6}

# Merge the two dictionaries
merged_dict = dict1.update(dict2)

# Print the merged dictionary
print(merged_dict)
```

Output:
```
{'a': 1, 
 'b': 2
 'c': 3, 
 'd': 4, 
 'e': 5
 'f': 6
}
```

In this example, `dict1` and `dict2` are merged into a new dictionary called `merged_dict`. The `update()` method takes another dictionary as an argument and merges its key-value pairs with the existing dictionary. The resulting dictionary is then printed.
----------------------------------------------------------------------------------------

Question 9: Code: How do I find all unique words in a sentence, ignoring case and punctuation?


Device set to use cpu


Cleaned Response:
```python
import re

def find_unique_words(sentence):
    # Remove punctuation and convert to lowercase
    cleaned_sentence = re.sub(r'[^\w\s]', '', sentence.lower())
    
    # Split the sentence into words
    words = cleaned_sentence.split()
    
    # Create a set of unique words
    unique_words = set(words)
    
    return list(unique_words)
```
----------------------------------------------------------------------------------------

Question 10: Code: How do I find the most frequent element in a list without using the collections module?


Device set to use cpu


Cleaned Response:
Here is a simple Python function that finds the most frequent element in an array without using the collections module.

```python
def most_frequent(arr):
    freq = {}
    max_freq = 0
    max_fq = None
    for i in arr:
        if i in freq:
            freq[i] += 1
        else:
            freq.update({i: 1})
        if freq[i] > max_freq:
            max_fq = i
            max_freq = freq[i]
    return max_fq
```

This function works by iterating over each element in the array and updating a dictionary `freq` to keep track of the count of each element. It also keeps track of the maximum frequency encountered so far. When it encounters an element that has a higher frequency than the current maximum, it updates the `max_fq` and `max_freq` variables. Finally, it returns the element with the highest frequency.
----------------------------------------------------------------------------------------

Question 11: Code: How do I determine if two strings are anagrams of 

In [18]:
# Cleanup extracted directory
try:
    shutil.rmtree(extract_dir)
    print(f"Successfully deleted directory: {extract_dir}")
except Exception as e:
    print(f"Error deleting {extract_dir}: {e}")

Successfully deleted directory: /home/flaniganp/Documents/my-python-buddy/models/llama-2-merged-7b-143k-codeAlpaca-2025-10-30_1143
